# Gen Z Consumer Insights - EDA

EDA on a synthetic 5,000-person Gen Z survey (`data/survey_responses.csv`), cross-checked against ~50 real cited industry stats (`data/industry_benchmarks.csv`). Class project, cleaning it up for GitHub.

**Other files:**
- `app.py` - same thing as a Streamlit dashboard
- `src/data_utils.py` - shared load/clean functions

**Sections**
1. [Setup](#1)
2. [Demographics](#2)
3. [Money & spending](#3)
4. [Media / screen time / platforms](#4)
5. [Values & trust](#5)
6. [Correlation check](#6)
7. [Key findings](#7)
8. [Data quality notes](#8)
9. [Outside sources](#9)


<a id='1'></a>
## 1. Setup

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_utils import load_survey, load_benchmarks, trust_gap_summary

sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.figsize"] = (9, 5)
pd.set_option("display.max_columns", 30)

survey = load_survey()
benchmarks = load_benchmarks()

print(f"Survey: {survey.shape[0]:,} rows x {survey.shape[1]} cols")
print(f"Benchmarks: {benchmarks.shape[0]} stats across {benchmarks.category.nunique()} categories")
survey.head()

In [ ]:
survey.info()

In [ ]:
survey.isnull().sum().to_frame("missing_values").T

Only missing column is `discretionary_pct_of_income`, 63 out of 5,000 rows (~1.3%). Looks like it's just respondents with $0 monthly income, so the % calc breaks (divide by zero). Not fixing it, just leaving it NaN and letting pandas skip it in the means.

In [ ]:
survey.describe().T

<a id='2'></a>
## 2. Demographics

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 9))

survey.region.value_counts().plot(kind="bar", ax=axes[0,0], color="#7C3AED")
axes[0,0].set_title("Region"); axes[0,0].tick_params(axis='x', rotation=0)

survey.gender.value_counts().plot(kind="bar", ax=axes[0,1], color="#7C3AED")
axes[0,1].set_title("Gender"); axes[0,1].tick_params(axis='x', rotation=30)

survey.education.value_counts().plot(kind="bar", ax=axes[1,0], color="#7C3AED")
axes[1,0].set_title("Education"); axes[1,0].tick_params(axis='x', rotation=20)

survey.employment.value_counts().plot(kind="bar", ax=axes[1,1], color="#7C3AED")
axes[1,1].set_title("Employment"); axes[1,1].tick_params(axis='x', rotation=20)

plt.tight_layout()
plt.savefig("../assets/demographics_overview.png", dpi=130, bbox_inches="tight")
plt.show()

Ages 18-28, so full adult Gen Z, not just teens. Region/gender spread look normal, nothing weird. Using employment as the main lens for the rest of this since it's the strongest predictor of income here, more than education or region.

<a id='3'></a>
## 3. Money & spending

In [ ]:
income_by_employment = survey.groupby("employment")["annual_income_usd"].agg(["mean","median","count"]).sort_values("mean")
income_by_employment.style.format({"mean": "${:,.0f}", "median": "${:,.0f}"})

Expected order - unemployed lowest, full-time highest. Sanity check passed.

In [ ]:
fig, ax = plt.subplots()
sample = survey.sample(1500, random_state=42)
sns.scatterplot(data=sample, x="annual_income_usd", y="monthly_discretionary_usd",
                 hue="employment", alpha=0.5, ax=ax, palette="Set2")
ax.set_title("Income vs. monthly discretionary spend")
ax.set_xlabel("Annual income ($)"); ax.set_ylabel("Monthly discretionary spend ($)")
plt.tight_layout()
plt.savefig("../assets/income_vs_discretionary.png", dpi=130, bbox_inches="tight")
plt.show()

print(f"r = {survey.annual_income_usd.corr(survey.monthly_discretionary_usd):.3f}")

In [ ]:
discretionary_pct_by_employment = survey.groupby("employment")["discretionary_pct_of_income"].mean().sort_values(ascending=False)
discretionary_pct_by_employment.round(1)

First actual finding. Discretionary spend tracks income almost 1:1 (r ~ 0.88), but as a % of income it's flat - 22.2 to 22.5% across every employment type. Unemployed, students, gig workers, full-time, doesn't matter. Employment status changes how much people make, not really how much of it they treat as spending money. Wasn't expecting it to be this flat.

In [ ]:
bnpl_by_income_q = pd.crosstab(survey.income_quartile, survey.uses_buy_now_pay_later, normalize="index") * 100
bnpl_by_income_q.round(1)

In [ ]:
fig, ax = plt.subplots()
bnpl_by_income_q["Yes"].plot(kind="bar", ax=ax, color="#F97316")
ax.set_title("BNPL adoption (%) by income quartile")
ax.set_ylabel("% using Buy Now, Pay Later")
ax.set_xlabel("Income quartile")
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig("../assets/bnpl_by_income.png", dpi=130, bbox_inches="tight")
plt.show()

BNPL usage sits at 41-43% no matter the income quartile. Assumed this would skew toward lower income since that's the whole "can't afford it upfront" narrative, but nope - looks like a general payment preference across the board, not an income thing.

<a id='4'></a>
## 4. Media, screen time & platforms

In [ ]:
fig, ax = plt.subplots()
sns.histplot(survey.daily_screen_hours, bins=40, color="#7C3AED", ax=ax)
ax.axvline(survey.daily_screen_hours.mean(), color="#F97316", linestyle="--",
           label=f"mean = {survey.daily_screen_hours.mean():.1f}h")
ax.set_title("Daily screen time distribution")
ax.set_xlabel("Hours / day")
ax.legend()
plt.tight_layout()
plt.savefig("../assets/screen_time_distribution.png", dpi=130, bbox_inches="tight")
plt.show()

print(survey.daily_screen_hours.describe().round(2))

Bell curve around 6 hours, long tail out to 16h for a few people. More on that in section 8.

In [ ]:
screen_by_platform = survey.groupby("primary_platform")["daily_screen_hours"].agg(["mean","count"]).sort_values("mean", ascending=False)
screen_by_platform.round(2)

Screen time is 5.9-6.2h regardless of platform. Reddit and Snapchat users actually report slightly more time than TikTok, which is the opposite of what I'd have guessed (everyone blames TikTok specifically). Difference is only ~0.3h though, basically noise.

Also: the ~6.0h average here is higher than the outside benchmark for social-media-only time (~4.5h/day, Cropink/Attest 2025). Makes sense, this column is total screen time, not just social apps, so it's not the same measurement.

In [ ]:
discovery = survey.brand_discovery_channel.value_counts(normalize=True).mul(100).round(1)
channel = survey.preferred_shopping_channel.value_counts(normalize=True).mul(100).round(1)

fig, axes = plt.subplots(1, 2, figsize=(13,5))
discovery.sort_values().plot(kind="barh", ax=axes[0], color="#7C3AED")
axes[0].set_title("How brands get discovered"); axes[0].set_xlabel("% of respondents")

channel.sort_values().plot(kind="barh", ax=axes[1], color="#F97316")
axes[1].set_title("Preferred shopping channel"); axes[1].set_xlabel("% of respondents")

plt.tight_layout()
plt.savefig("../assets/discovery_and_channel.png", dpi=130, bbox_inches="tight")
plt.show()

Social media is the top brand discovery channel at 40.8%, beats search (24.1%), word of mouth (17.8%), influencer content (11.0%), and traditional ads dead last (6.3%). Mobile apps also win for shopping channel (39.3%). Both line up with the outside benchmarks in section 9.

<a id='5'></a>
## 5. Values & trust

In [ ]:
tg = trust_gap_summary(survey)
fig, ax = plt.subplots(figsize=(7,3.5))
ax.barh(tg.channel, tg.avg_trust_1to5, color=["#94A3B8", "#7C3AED"])
for i, v in enumerate(tg.avg_trust_1to5):
    ax.text(v + 0.05, i, f"{v:.2f}", va="center")
ax.set_xlim(0, 5)
ax.set_title("Avg trust: traditional ads vs influencers (1-5)")
plt.tight_layout()
plt.savefig("../assets/trust_gap.png", dpi=130, bbox_inches="tight")
plt.show()

In [ ]:
for group_col in ["region", "gender", "employment", "income_quartile"]:
    print(f"\n--- {group_col} ---")
    g = survey.groupby(group_col)[["trust_traditional_ads_1to5","trust_influencers_1to5"]].mean()
    g["gap"] = g["trust_influencers_1to5"] - g["trust_traditional_ads_1to5"]
    print(g.round(2))

Strongest signal in the dataset. Ad trust 2.33/5, influencer trust 3.39/5, roughly a full point gap, and it barely moves across region, gender, employment, or income (all land between 1.00 and 1.14). Only outlier is the "prefer not to say" gender group (n=45, gap widens to 1.49), but that sample's too small to read much into. Something this consistent across every subgroup usually means it's real, not a fluke in one slice.

<a id='6'></a>
## 6. Correlation check

In [ ]:
numeric_cols = ["annual_income_usd","monthly_discretionary_usd","daily_screen_hours",
                 "val_sustainability_1to5","val_brand_authenticity_1to5",
                 "trust_traditional_ads_1to5","trust_influencers_1to5"]

corr = survey[numeric_cols].corr()
fig, ax = plt.subplots(figsize=(8,6))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="RdPu", center=0, ax=ax, square=True)
ax.set_title("Correlation matrix")
plt.tight_layout()
plt.savefig("../assets/correlation_heatmap.png", dpi=130, bbox_inches="tight")
plt.show()

Aside from income vs discretionary spend (0.88), everything else is under +-0.05. Screen time doesn't predict trust in influencers. Sustainability and brand authenticity don't move together. Basically none of the attitude questions correlate with anything.

Probably means these Likert fields are drawn independently on top of the demographics, which happens a lot in synthetic survey data. Doesn't make the dataset useless, just means I shouldn't invent "segment X believes Y because of Z" stories the correlations don't back up. The trust gap in section 5 still holds because it's a difference in averages, not a claimed correlation.

<a id='7'></a>
## 7. Key findings

1. Ad trust vs influencer trust gap (2.33 vs 3.39/5) - most consistent finding, holds across every subgroup.
2. Discretionary spend is a fixed ~22% of income no matter employment type.
3. BNPL usage (~42%) flat across income quartiles.
4. Social media is the top brand discovery channel (40.8%).
5. Screen time barely differs by platform (5.9-6.2h).

<a id='8'></a>
## 8. Data quality notes

- Attitude variables basically don't correlate with anything (section 6) - don't over-read the findings above as causal.
- 9 respondents (0.18%) report screen time 3+ SDs above the mean, up to 16h/day. Could be real, could be double-counted devices. Wouldn't trust it for anything beyond exploratory work.
- Non-binary (n=92) and prefer not to say (n=45) gender groups are small - any differences involving them are directional at best.
- Survey's screen time (~6.0h) vs outside social-only benchmark (~4.5h) aren't measuring the same thing, so don't compare them directly.

In [ ]:
benchmarks.head(10)

<a id='9'></a>
## 9. Outside sources

The low ad trust and income-driven spending findings line up with real research on Gen Z. Numbers below are also logged in `data/industry_benchmarks.csv`:

- Gen Z global spending power projected to roughly 4x, ~$2.7T (2024) to ~$12.6T (2030). [Bank of America Institute, 2025](https://institute.bankofamerica.com/economic-insights/genz-new-economic-force.html)
- US Gen Z spending dropped ~13% Jan-Apr 2025, mostly apparel/accessories/electronics. [PwC, 2025](https://www.pwc.com/us/en/industries/consumer-markets/library/gen-z-consumer-trends.html)
- Weekly video game spending for 18-24 year olds down ~25% YoY vs under 5% for older generations. [via PC Gamer, 2025](https://www.pcgamer.com/gaming-industry/new-study-shows-that-gen-z-is-spending-way-less-money-on-videogames-than-older-gamers/)
- Average credit card balance ~$3,493 as of mid-2025, lowest of any adult generation. [Experian, 2025](https://www.experian.com/blogs/ask-experian/research/credit-card-debt-by-age/)
- 85% say social media influences what they buy. [Retail Dive, 2025](https://www.retaildive.com/news/generation-z-social-media-influence-shopping-behavior-purchases-tiktok-instagram/652576/)

Basically: big long-run spending power, low trust in traditional ads, but currently spending more cautiously than the projections suggest. Marketing to this group probably means creator/peer channels over traditional ads.

Side note: some rows in `data/industry_benchmarks.csv` have conflicting numbers across sources (QSR visit trends especially). Left the disagreement in instead of picking one number.